In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'tensorflow', 'keras', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'keras': 'keras==3.14.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "591ab4e3a3e575d95d805a92550ef076fd65a824"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/591ab4e3a3e575d95d805a92550ef076fd65a824/d2l"
for name in ('__init__.py', 'tensorflow.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Implementation of Multilayer Perceptrons

An MLP implementation extends a linear classifier by composing affine layers
with nonlinear activations. We first write this composition explicitly and
then use the corresponding framework layers.

In [ ]:
from d2l import tensorflow as d2l
import tensorflow as tf

## Implementation from Scratch

We begin with an implementation using tensors and automatic differentiation.

### Initializing Model Parameters

Recall that Fashion-MNIST contains 10 classes,
and that each image consists of a $28 \times 28 = 784$
grid of grayscale pixel values.
As before we will disregard the spatial structure
among the pixels for now,
so we can think of this as a classification dataset
with 784 input features and 10 classes.
To begin, we will implement an MLP
with one hidden layer and 256 hidden units
(the figure).
Both the number of layers and their width are adjustable
(they are considered hyperparameters).
Powers-of-two widths often align well with accelerator kernels, although the
fastest dimensions depend on the hardware and numerical precision.

![The two-layer MLP of this section: a batched input is flattened to 784 features, mapped by an affine layer to a 256-dimensional hidden representation, passed through a ReLU, then mapped by a second affine layer to 10 logits.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-mlp-arch.svg)

Again, we will represent our parameters with several tensors.
Note that *for every layer*, we must keep track of
one weight matrix and one bias vector.
As always, we allocate memory
for the gradients of the loss with respect to these parameters.
We use small Gaussian noise ($\sigma = 0.01$) as a simple starting point;
principled strategies for choosing this scale are the subject of
that section.

In the code below we use `tf.Variable`
to define the model parameter.

In [ ]:
class MLPScratch(d2l.Classifier):
    def __init__(self, num_inputs, num_outputs, num_hiddens, lr, sigma=0.01):
        super().__init__()
        self.save_hyperparameters()
        self.W1 = tf.Variable(
            tf.random.normal((num_inputs, num_hiddens)) * sigma)
        self.b1 = tf.Variable(tf.zeros(num_hiddens))
        self.W2 = tf.Variable(
            tf.random.normal((num_hiddens, num_outputs)) * sigma)
        self.b2 = tf.Variable(tf.zeros(num_outputs))

### Model

To expose the complete forward computation, we implement ReLU directly rather
than invoking the built-in `relu` function.

In [ ]:
def relu(X):
    return tf.math.maximum(X, 0)

Since we are disregarding spatial structure,
we `reshape` each two-dimensional image into
a flat vector of length  `num_inputs`.
The forward method then requires only the flattening and two affine
computations. Automatic differentiation supplies the backward pass.

In [ ]:
@d2l.add_to_class(MLPScratch)
def forward(self, X):
    X = d2l.reshape(X, (-1, self.num_inputs))
    H = relu(d2l.matmul(X, self.W1) + self.b1)
    return d2l.matmul(H, self.W2) + self.b2

### Training

The training loop is the same as for softmax regression. We define the model,
data, and trainer, then invoke `fit` on the model and data.

In [ ]:
model = MLPScratch(num_inputs=784, num_outputs=10, num_hiddens=256, lr=0.1)
data = d2l.FashionMNIST(batch_size=256)
trainer = d2l.Trainer(max_epochs=30)
trainer.fit(model, data)

In the displayed run, validation accuracy settles near $0.87$, above the earlier
softmax-regression run. This comparison is illustrative: initialization,
shuffling, framework defaults, and optimization settings also affect the result.

## Concise Implementation

High-level framework layers provide a more concise implementation of the same
architecture.

### Model

Compared with our concise implementation
of softmax regression
(that section),
the only difference is that we add
*two* fully connected layers where we previously added only *one*.
The first is the hidden layer,
the second is the output layer.

In [ ]:
class MLP(d2l.Classifier):
    def __init__(self, num_outputs, num_hiddens, lr):
        super().__init__()
        self.save_hyperparameters()
        self.net = tf.keras.models.Sequential([
            tf.keras.layers.Flatten(),
            tf.keras.layers.Dense(num_hiddens, activation='relu'),
            tf.keras.layers.Dense(num_outputs)])

A `forward` method applies each model transformation to the output of the
preceding one. Here, `MLP` inherits the `Module` implementation
(that section), which invokes `self.net(X)`. The
`Sequential` container applies its registered transformations in order and
therefore defines the forward computation. that section
examines this container in detail.


### Training

The training loop is the same as for softmax regression.
This modularity enables us to separate
matters concerning the model architecture
from orthogonal considerations.
Note that while the two versions compute the same *architecture*, they do not
start from the same *parameters*: the scratch model draws its weights from
$\mathcal{N}(0, 0.01^2)$, whereas the concise version uses the library's
default initializer. Defaults differ across libraries and layer types, so the
two versions need not start at comparable scales. Their training trajectories
and final accuracies can therefore differ.

In [ ]:
model = MLP(num_outputs=10, num_hiddens=256, lr=0.1)
trainer.fit(model, data)

## Summary

We have built and trained a multilayer perceptron in both from-scratch and concise
forms. The from-scratch version exposes two weight matrices, two bias vectors, a
ReLU, and the two affine computations. The concise version represents the same
architecture as a four-element `nn.Sequential` stack. The training loop, loss,
and data loader remain unchanged from softmax regression.

The from-scratch version also shows why high-level containers are useful:
manually naming and tracking parameters becomes cumbersome as layers are added
or reordered. `nn.Sequential` handles both registration and execution order.

Three questions remain open, and each is the subject of one of the next sections:

* **How do gradients flow through this stack, and what can go wrong as it gets deeper?** (that section, that section)
* **Why does such a flexible model generalize to unseen data at all?** (that section)
* **How can we regularize it to generalize better?** (that section)

The following sections address these requirements for reliable training.


## Exercises

1. [code] **Width and learning rate.** Train `MLP` for 10 epochs at every
   combination of learning rate in $\{0.01, 0.03, 0.1, 0.3\}$ and hidden
   width `num_hiddens` in $\{32, 64, 128, 256\}$, and plot the final
   validation accuracy as a heatmap.
    1. At the learning rate $0.1$ used in this section, which width gives the
       best result, and how strongly does accuracy depend on width there?
    1. Does the best learning rate shift systematically with width?
    1. Repeat the sweep over learning rates at width 256 with 20 epochs
       instead of 10. How does the best learning rate relate to the training
       budget?
1. **Parameter count.** The one-hidden-layer model of this section maps 784
   inputs through $h$ hidden units to 10 outputs.
    1. Give a closed-form expression for the total number of parameters as a
       function of $h$.
    1. At which $h$ does the parameter count equal the number of training
       examples in Fashion-MNIST (60,000)? Compare this value with the 256
       units used in this section.
1. **Single hidden unit.** Why is it a bad idea to insert a hidden layer with
   a single neuron? What could go wrong?
1. [extended] **Joint hyperparameter search.** Hyperparameters interact: the
   best value of one depends on the values of the others.
    1. Optimize over all hyperparameters jointly: learning rate, number of
       epochs, number of hidden layers, and number of hidden units per layer.
       What is the best result you can achieve?
    1. Why is tuning several hyperparameters jointly much harder than tuning
       each one in isolation?
    1. Describe an efficient strategy for searching over multiple
       hyperparameters jointly.
1. [code] **Gradient check.** Approximate the derivative of the loss with
   respect to an entry $w$ of `W1` in `MLPScratch` by a central difference,

    $$\frac{\partial \ell}{\partial w} \approx \frac{\ell(w + \epsilon) - \ell(w - \epsilon)}{2\epsilon},$$

    evaluated on a single minibatch, and compare the result with the gradient
    computed by automatic differentiation. Do this for a random sample of 50
    entries of `W1` and report the maximum relative error over the sample. In
    double precision a correct implementation stays below $10^{-5}$.

    *Adapted from Stanford CS231n,
    [Assignment 1](https://cs231n.github.io/assignments2023/assignment1/),
    two-layer network.*
1. [code] **Benchmarking implementations.** Compare the speed of `MLPScratch`
   and `MLP`: train both at hidden widths 256, 1024, and 4096 for three
   epochs each on Fashion-MNIST, recording the wall-clock time per epoch.
   Plot the ratio of `MLPScratch` time to `MLP` time as a function of width.
   Does the ratio grow, shrink, or stay flat as the network gets larger?
   Suggest an explanation.
1. [code] **Memory alignment.** Measure the speed of tensor--matrix
   multiplications for well-aligned and misaligned matrices, for instance
   with dimensions 1024, 1025, 1026, 1028, and 1032.
    1. How do the results differ between GPUs and CPUs?
    1. Determine the memory bus width of your CPU and GPU.
1. [code] **Activation functions.** Compare at least ReLU, tanh, sigmoid, and
   GELU as the hidden activation of `MLP` on Fashion-MNIST, retuning the
   learning rate for sigmoid and tanh. Which works best, and by how much?
   GELU is used in BERT and GPT-2-style Transformers, while many recent
   language models use gated SiLU (SwiGLU) blocks. State what a comparison
   would have to control for before its result could guide the choice of
   activation in a Transformer.
1. [code] **Initialization scale and depth.** `MLPScratch` initializes its
   weights with Gaussian noise of standard deviation $\sigma = 0.01$.
    1. Train the one-hidden-layer model with $\sigma = 0.001$, $0.01$, and
       $0.1$, and plot the training and validation curves for each. Why does
       $\sigma$ matter? Consider what happens to the activations on the first
       forward pass.
    1. Add a second hidden layer of width 256, keeping $\sigma = 0.01$, and
       compare its training curve with that of the one-hidden-layer model.
    1. Before running anything, predict which of a two-hidden-layer and a
       four-hidden-layer version, each hidden layer of width 256, is more
       sensitive to $\sigma$, and why. Then verify: sweep $\sigma$ over
       $\{0.001, 0.003, 0.01, 0.03, 0.1\}$ for both depths and report the
       range of $\sigma$ for which each network reaches at least 70% training
       accuracy within five epochs. that section
       develops the principled answer.

    *Adapted from Stanford CS231n,
    [Assignment 2](https://cs231n.github.io/assignments2023/assignment2/),
    FullyConnectedNets Inline Question 1.*

[Discussions](https://d2l.discourse.group/t/227)